In [1994]:
import struct
import random

# 32 + 32 + 2 + 1 + 5 = 72 bytes total
_PRIVATE_RULE_SPEC_SIZE = 72

def pack_spec(mask_pre: int, mask_post: int, reps: int, flip: bool) -> bytes:
    pre_bytes = mask_pre.to_bytes(32, 'big')
    post_bytes = mask_post.to_bytes(32, 'big')
    reps_bytes = struct.pack('>H', reps)   # 2 bytes
    flip_bytes = struct.pack('>?', flip)   # 1 byte
    padding = b'\x00' * 5                  # 5 bytes padding to match original layout

    return pre_bytes + post_bytes + reps_bytes + flip_bytes + padding

def unpack_spec(packed: bytes) -> tuple[int, int, int, bool]:
    if len(packed) != _PRIVATE_RULE_SPEC_SIZE:
        raise ValueError(f"Expected {_PRIVATE_RULE_SPEC_SIZE} bytes, got {len(packed)}")

    pre_bytes = packed[0:32]
    post_bytes = packed[32:64]
    reps = struct.unpack('>H', packed[64:66])[0]
    flip = struct.unpack('>?', packed[66:67])[0]
    # padding = packed[67:72]  # ignored

    mask_pre = int.from_bytes(pre_bytes, 'big')
    mask_post = int.from_bytes(post_bytes, 'big')

    return mask_pre, mask_post, reps, flip

# ---------- bit helpers ----------
def parity(x: int) -> int:              # faster than while-loop
    return bin(x).count('1') & 1

def extract_window_finite(state: int, i: int, L: int, N: int, pad_bit: int = 0) -> int:
    """Return the N-bit window that **starts** at absolute position `i`."""
    w = 0
    for j in range(N):
        idx = i + j
        bit = ((state >> idx) & 1) if 0 <= idx < L else pad_bit
        w = (w << 1) | bit
    return w

def apply_rule_expand(state: int, rule: int, L: int, N: int, pad_bits: int = 0) -> tuple[int, int]:
    grow   = N - 1
    new_L  = L + grow
    padded = (state << grow) | (pad_bits & ((1 << grow) - 1))

    out = 0
    for i in range(new_L):
        w     = extract_window_finite(padded, i, new_L, N)
        bit_i = (rule >> w) & 1
        out  |= bit_i << i
    return out, new_L

def implements_spec(pre: int, post: int, spec: bytes) -> bool:
    mask_pre, mask_post, _, flip = unpack_spec(spec)
    return parity(mask_pre & pre) ^ flip == parity(mask_post & post)


def random_bitmask(N: int, P: int) -> int:
    if N < 0 or P < 0:
        raise ValueError("N and P must be non-negative")
    if P > N:
        raise ValueError("P cannot exceed N")

    positions = random.sample(range(N), P)   # choose P distinct bit positions

    mask = 0
    for pos in positions:
        mask |= (1 << pos)

    return mask


def random_rule(N: int) -> int:
    return random.randrange(1 << N)


def get_resulting_len(L_start, N, M):
    return L_start + M*(N - 1)

def get_random_spec1(L_start, N, M):
    L = get_resulting_len(L_start, N, M)
    mask_pre = random_bitmask(L_start, 1)
    mask_post = random_bitmask(L, 1)
    reps = M
    flip = random.choice([True, False])
    return pack_spec(mask_pre, mask_post, reps, flip)

def get_random_spec2(L_start, N, M):
    L = get_resulting_len(L_start, N, M)
    mask_pre = random_bitmask(L_start, 2)
    mask_post = random_bitmask(L, 2)
    reps = M
    flip = random.choice([True, False])
    return pack_spec(mask_pre, mask_post, reps, flip)

def flip_spec(spec: bytes) -> bytes:
    mask_pre, pask_post, reps, flip = unpack_spec(spec)
    return pack_spec(mask_pre, pask_post, reps, flip)

In [2820]:
def get_spec_matching_rule(L_start: int, N: int, spec: bytes, max_rules: int = 2**10) -> tuple[Union[int|None], bool]:
    mask_pre, mask_post, reps, flip = unpack_spec(spec)
    flip_spec = pack_spec(mask_pre, mask_post, reps, not flip)

    rules = list(range(1 << (1 << N)))
    random.shuffle(rules)
    rules = rules[:max_rules]

    for rule in rules:
        pad = (1 << N) - 1
        ok = True
        ok_flip = True

        for pre in range(1 << L_start):
            curr, curr_L = pre, L_start

            for _ in range(reps):
                curr, curr_L = apply_rule_expand(curr, rule, curr_L, N, pad)

            mask_pre, mask_post, reps, flip = unpack_spec(spec)

            if ok and not implements_spec(pre, curr, spec):
                ok = False

            if ok_flip and not implements_spec(pre, curr, flip_spec):
                ok_flip = False

            if not ok and not ok_flip:
                break

        if ok or ok_flip:
            return rule, ok_flip

    return None, False

def generate_spec_rule_pair(L_start: int, N: int, M: int) -> tuple[bytes, int]:
    for _ in range(10):
        spec = get_random_spec2(L_start, N, M)
        rule, flip = get_spec_matching_rule(L_start, N, spec)

        if not rule:
            continue

        if flip:
            spec = flip_spec(spec)

        return spec, rule

    assert False, "No matching spec found"

def spec_string(spec: bytes, L_start: int, L: int):
    mask_pre, mask_post, reps, flip = unpack_spec(spec)
    return (f" pre: {mask_pre:0{L_start}b}\n"
        + f"post: {mask_post:0{L}b}\n"
        + f"reps: {reps}\n"
        + f"flip: {flip}")

pairs = []

N = 4
L_start = N
M = 3

L_starts = [get_resulting_len(L_start, N, M), N]

spec, rule = generate_spec_rule_pair(L_starts[0], N, M)
mask_post_next, _, _, _ = unpack_spec(spec)
print(f"mask_post:{mask_post:0{L_starts[0]}b}")
pairs.append((L_start, get_resulting_len(L_start, N, M), spec, rule))

for L_start in L_starts[1:]:
    ok = False

    for i in range(50):
        if L_start == L_starts[-1]:
            mask_pre = random_bitmask(L_start-1, 1) | (1 << (L_start-1))
        else:
            mask_pre = random_bitmask(L_start, 2)

        print(f"mask_post:{mask_post:0{L_starts[0]}b}")
        spec = pack_spec(mask_pre, mask_post_next, M, random.choice([False, True]))
        rule, flip = get_spec_matching_rule(L_start, N, spec)

        if not rule:
            continue

        if flip:
            spec = flip_spec(spec)

        pairs.append((L_start, get_resulting_len(L_start, N, M), spec, rule))
        ok = True
        break

    assert ok is True

pairs.reverse()
print([(l1, l2, spec_string(spec, l1, l2), rule) for (l1, l2, spec, rule) in pairs])

mask_post:100000000000001000000
mask_post:100000000000001000000
[(4, 13, ' pre: 1100\npost: 0100000000001\nreps: 3\nflip: True', 51550), (4, 13, ' pre: 100000000001\npost: 100000000000000001\nreps: 3\nflip: True', 31322)]


In [3076]:
def encode_bit(L_start: int, N: int, bit: int, pairs: List[tuple[int, int, bytes, int]]) -> tuple[bytes, int]:
    key_len = N-1
    key = random.getrandbits(key_len)
    print(f" key: {key:03b}")
    bits = (bit << (N-1)) | key
    print(f"bits:{bits:04b}")
    pad = (1 << (N - 1)) - 1
    print(f" pad:{pad:03b}")

    curr, curr_L = bits, L_start

    for (l1, l2, spec, rule) in pairs:
        _, _, reps, _ = unpack_spec(spec)
        before = curr
        print("Rule:", rule)
        for _ in range(reps):
            curr, curr_L = apply_rule_expand(curr, rule, curr_L, N, pad)
        print("SPEC:", spec_string(spec, l1, l2))
        print(f"Implements? {implements_spec(before, curr, spec)}")
        print()

    return curr, curr_L, key


def decode_bit(final_state: int, key: int, pairs: List[tuple[int,int,bytes,int]]) -> int:
    """
    Recover the payload bit by reading the final spec’s mask_post and
    then XORing all of the flip‐flags in sequence.
    """
    # last spec in list
    len_pre, L, spec_last, _ = pairs[-1]
    mask_pre, mask_post_last, _, flip_last = unpack_spec(spec_last)
    print(f"last mask: {mask_post_last:0{L}b}")
    print(f"masked   : {mask_post_last & final_state:0{L}b}")

    # start with parity of the final mask_post
    p = parity(mask_post_last & final_state)
    print(f"parity   : {p:01b}")
    p ^= flip_last
    print(f"flip ({flip_last}): {p:01b}")

    # then undo each earlier flip
    for (len_pre, _, spec, _) in pairs[:-1]:
        mask_pre, _, _, flip = unpack_spec(spec)
        p ^= flip
        print(f"flip ({flip}): {p:01b}")

    bits = (1 << (len_pre-1)) | key

    return p ^ parity(bits & mask_pre) ^ 1

b = random.getrandbits(1)
e, l, key = encode_bit(N, N, b, pairs)
print(f"{e:0{l}b}")
d = decode_bit(e, key, pairs)
print(b, d, d == b)

 key: 010
bits:1010
 pad:111
Rule: 51550
SPEC:  pre: 1100
post: 0100000000001
reps: 3
flip: True
Implements? True

Rule: 31322
SPEC:  pre: 100000000001
post: 100000000000000001
reps: 3
flip: True
Implements? True

0001111011100110011100
last mask: 100000000000000001
masked   : 100000000000000000
parity   : 1
flip (True): 0
flip (True): 1
1 1 True


In [93]:
def extract_window_right_pad(state: int, i: int, L: int, N: int, pad_bits: int) -> int:
    w = 0

    for j in range(N):
        idx = i + j

        if idx < L:
            bit = (state >> idx) & 1
        else:
            bit = (pad_bits >> (idx - L)) & 1

        w = (w << 1) | bit
    return w


def apply_rule_append_pad(state: int,
                          rule: int,
                          L: int,
                          N: int,
                          pad_bits: int) -> tuple[int, int]:
    grow  = N - 1
    new_L = L + grow
    out   = 0

    state = state << grow
    grow_mask = (1 << grow) - 1
    state |= pad_bits & grow_mask

    for i in range(L):
        w     = extract_window_finite(state, i, L, N)
        bit_i = (rule >> w) & 1
        out  |= bit_i << i

    return out, new_L

# Rule-30, N = 4 → grow = 3 (window length 4, right-expanding)
rule30   = 0b10111110
state, L = 0b1, 1          # initial single '1'
N        = 3
pad_bits = 0b01           # bits “101” will be glued onto the right each step

for t in range(5):
    state, L = apply_rule_append_pad(state, rule30, L, N, pad_bits)
    print(f"{state:0{L}b}")


001
00101
0010101
001010101
00101010101
